### Parsing the full log data to structure it with file names as Y-axis and categories as X-axis


In [2]:
import pandas as pd

In [3]:
full_log_entries = """
log6111:insertion_time_ns = 29380621649
log6111:rd_time_ns = 5947611
log6111:total_cpu_time_ns = 7000000.00
log6111:whole block total_cpu_time_ns = 12997000.00
log6111:fixed #PQ = 5000 point_query_time_on_existing_keys_ns = 13987821739
log6111:fixed #PQ = 100000 point_query_time_on_currently_deleted_all_ns = 189712279953
log6111:all_time_ns = 644714421925
log6211:insertion_time_ns = 24549495963
log6211:rd_time_ns = 4301581
log6211:total_cpu_time_ns = 6998000.00
log6211:whole block total_cpu_time_ns = 12999000.00
log6211:fixed #PQ = 5000 point_query_time_on_existing_keys_ns = 11007880740
log6211:fixed #PQ = 100000 point_query_time_on_currently_deleted_all_ns = 133721857650
log6211:all_time_ns = 593309641205
log6311:insertion_time_ns = 28476223999
log6311:rd_time_ns = 5647276
log6311:total_cpu_time_ns = 9001000.00
log6311:whole block total_cpu_time_ns = 12999000.00
log6311:fixed #PQ = 5000 point_query_time_on_existing_keys_ns = 10930288698
log6311:fixed #PQ = 100000 point_query_time_on_currently_deleted_all_ns = 81322825952
log6311:all_time_ns = 530381032613
log6411:insertion_time_ns = 27936218462
log6411:rd_time_ns = 6256085
log6411:total_cpu_time_ns = 5001000.00
log6411:whole block total_cpu_time_ns = 12999000.00
log6411:fixed #PQ = 5000 point_query_time_on_existing_keys_ns = 10759425464
log6411:fixed #PQ = 100000 point_query_time_on_currently_deleted_all_ns = 593936887
log6411:all_time_ns = 445625820413
log6511:insertion_time_ns = 27134548472
log6511:rd_time_ns = 6209362
log6511:total_cpu_time_ns = 6000000.00
log6511:whole block total_cpu_time_ns = 12999000.00
log6511:fixed #PQ = 5000 point_query_time_on_existing_keys_ns = 11168430757
log6511:fixed #PQ = 100000 point_query_time_on_currently_deleted_all_ns = 138128109695
log6511:all_time_ns = 589780996068
log6611:insertion_time_ns = 29080085282
log6611:rd_time_ns = 5876848
log6611:total_cpu_time_ns = 7997000.00
log6611:whole block total_cpu_time_ns = 12998000.00
log6611:fixed #PQ = 5000 point_query_time_on_existing_keys_ns = 11339916931
log6611:fixed #PQ = 100000 point_query_time_on_currently_deleted_all_ns = 88775115581
log6611:all_time_ns = 534545849795
log6711:insertion_time_ns = 29328399156
log6711:rd_time_ns = 5772280
log6711:total_cpu_time_ns = 5001000.00
log6711:whole block total_cpu_time_ns = 12999000.00
log6711:fixed #PQ = 5000 point_query_time_on_existing_keys_ns = 11187330400
log6711:fixed #PQ = 100000 point_query_time_on_currently_deleted_all_ns = 140622226889
log6711:all_time_ns = 586711948424
log6811:insertion_time_ns = 29347191017
log6811:rd_time_ns = 5784828
log6811:total_cpu_time_ns = 4999000.00
log6811:whole block total_cpu_time_ns = 12999000.00
log6811:fixed #PQ = 5000 point_query_time_on_existing_keys_ns = 11325089147
log6811:fixed #PQ = 100000 point_query_time_on_currently_deleted_all_ns = 103673661355
log6811:all_time_ns = 548566971595
log6911:insertion_time_ns = 29325392715
log6911:rd_time_ns = 6122608
log6911:total_cpu_time_ns = 8000000.00
log6911:whole block total_cpu_time_ns = 12999000.00
log6911:fixed #PQ = 5000 point_query_time_on_existing_keys_ns = 11329017144
log6911:fixed #PQ = 100000 point_query_time_on_currently_deleted_all_ns = 140785713468
log6911:all_time_ns = 588174015515
log7011:insertion_time_ns = 28889132084
log7011:rd_time_ns = 5945439
log7011:total_cpu_time_ns = 8000000.00
log7011:whole block total_cpu_time_ns = 12999000.00
log7011:fixed #PQ = 5000 point_query_time_on_existing_keys_ns = 11314145473
log7011:fixed #PQ = 100000 point_query_time_on_currently_deleted_all_ns = 88812435407
log7011:all_time_ns = 534438654359
"""

In [4]:

# Creating a dictionary to store structured data
structured_data = {}

# Process each log entry
for line in full_log_entries.strip().split("\n"):
    parts = line.split(" = ")
    if len(parts) == 2:
        log_id, metric = parts[0].split(":", 1)
        value = float(parts[1]) if "." in parts[1] else int(parts[1])
        
        if log_id not in structured_data:
            structured_data[log_id] = {}
        
        structured_data[log_id][metric] = value
    
    if len(parts) == 3:
        log_id, metric_1 = parts[0].split(":", 1)
        metric = metric_1 + " = " + parts[1]
        # print(log_id, metric)
        value = float(parts[2]) if "." in parts[2] else int(parts[2])
        
        if log_id not in structured_data:
            structured_data[log_id] = {}
        
        structured_data[log_id][metric] = value


In [17]:

# Define the required order based on the 4-digit prefix after "log"
required_order = [11, 21, 31, 41, 51, 61, 71, 81, 91, 101]
offset = 6000
n_group = 1

# print(str(list(structured_data.keys())))
filenames = list(structured_data.keys())

# Reinitialize a dictionary for ordered storage based on the correct mapping
sorted_filenames_corrected = {key: [] for key in required_order}

# Process each filename and categorize based on the given order
for filename in filenames:
    # num_part = (int(filename[3:7]) % 10)
    num_part = (int(filename[3:7]) - offset) // 100 * 10 + (int(filename[3:7]) % 100) // 10
    if num_part in sorted_filenames_corrected:
        sorted_filenames_corrected[num_part].append(filename)

# sorted_filenames_corrected = 
file_names_list_2d = [list(val) for val in sorted_filenames_corrected.values()]
file_names_list_2d = list(zip(*file_names_list_2d))  # Correct transposition



print(file_names_list_2d)
# # Ensure ordering is correct by maintaining the specified order
# df_sorted_filenames = pd.DataFrame.from_dict(sorted_filenames_corrected, orient="index").T

# # Display the corrected sorted groups
# tools.display_dataframe_to_user(name="Reordered Log Groups (Final)", dataframe=df_sorted_filenames)



[('log6111', 'log6211', 'log6311', 'log6411', 'log6511', 'log6611', 'log6711', 'log6811', 'log6911', 'log7011')]


In [18]:

# Convert dictionary to DataFrame
df_structured = pd.DataFrame.from_dict(structured_data, orient="index")

# Reset index and rename columns
df_structured.reset_index(inplace=True)
df_structured.rename(columns={"index": "File Name"}, inplace=True)



print(df_structured["File Name"])

0    log6111
1    log6211
2    log6311
3    log6411
4    log6511
5    log6611
6    log6711
7    log6811
8    log6911
9    log7011
Name: File Name, dtype: object


In [22]:
# # Initialize a dictionary to store the new structured data
# new_df_table = {}

# # Iterate through each group ID in sorted order
# for file_group in file_names_list_2d:
#     for log_id in file_group:
#         # Extract the corresponding row from df_structured
#         line = df_structured[df_structured["File Name"] == log_id]
#         # print(log_id)
#         print(line)
#         # Store it in the new table
#         new_df_table[log_id] = line

# # # Convert dictionary to a structured DataFrame
# df_new_table = pd.concat(new_df_table.values())
# # df_new_table = new_df_table
# # df_new_table = pd.DataFrame.from_dict(new_df_table.values(), orient="index")
df_new_table = df_structured

# Save the new structured table
structured_table_path = "./structured_log_data.xlsx"
df_new_table.to_excel(structured_table_path, index=False)

print(df_new_table)
# # Provide download link
# structured_table_path


  File Name  insertion_time_ns  rd_time_ns  total_cpu_time_ns   
0   log6111        29380621649     5947611          7000000.0  \
1   log6211        24549495963     4301581          6998000.0   
2   log6311        28476223999     5647276          9001000.0   
3   log6411        27936218462     6256085          5001000.0   
4   log6511        27134548472     6209362          6000000.0   
5   log6611        29080085282     5876848          7997000.0   
6   log6711        29328399156     5772280          5001000.0   
7   log6811        29347191017     5784828          4999000.0   
8   log6911        29325392715     6122608          8000000.0   
9   log7011        28889132084     5945439          8000000.0   

   whole block total_cpu_time_ns   
0                     12997000.0  \
1                     12999000.0   
2                     12999000.0   
3                     12999000.0   
4                     12999000.0   
5                     12998000.0   
6                     12999000.0

In [10]:


# display(df_structured)

# # Save structured data to Excel
# #structured_excel_filename = "/mnt/data/structured_log_data.xlsx"
# structured_excel_filename = "./structured_log_data.xlsx"
# # structured_excel_filename = "./structured_log_data.csv"
# df_structured.to_excel(structured_excel_filename, index=False)

# ## Provide download link
# #structured_excel_filename

